# Data management

A model reads **windows** of its signals: the last 10 samples of a throttle,
the next sample of a position, a trajectory of 50 steps. Recorded data is
instead one long array per signal. The `DataLoader` cuts the recordings into
the samples the model reads, every input aligned on the same instant.

This tutorial shows:

- how the windows are cut and aligned, on a signal you can follow by eye;
- every kind of source: arrays, DataFrames, CSV files, folders;
- several recordings (simulations), train/validation splits, sub-sampling;
- sequences, dynamic lengths and padding;
- missing values, short recordings, and data that is already windowed;
- how to read the samples back, and feed them to a model.

The `DataLoader` reads the windows and sequences from the model, so the model is
declared and **built** first ([Model design](01_model_design.ipynb)).

In [1]:
import os

# Pick the Keras backend before nnodely (or Keras) is imported:
# "tensorflow" (the default), "torch" or "jax".
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import tempfile
from pathlib import Path

import keras
import numpy as np
import pandas as pd

from nnodely import (
    DataLoader,
    Fir,
    Input,
    Linear,
    Modely,
    Output,
    Parameter,
    TimeRange,
    TimeSelect,
    set_seed,
)

set_seed(0)
workdir = Path(tempfile.mkdtemp(prefix="nnodely_data_"))  # where the CSV files go

## How the windows are cut

Three inputs, each with a different window:

- `a` reads its last 3 samples, `tw=3`;
- `b` its current sample and the next two, `tw=(1, 2)`;
- `c` one value per sample, no window.

The signals are counters, so each value tells where it comes from: `a` holds
`0, 1, 2, ...`, `b` holds `100, 101, ...` and `c` holds `1000, 1001, ...`.

In [2]:
a = Input("a", tw=3)
b = Input("b", tw=(1, 2))
c = Input("c")
out = Output("out", Fir(features=1)([a]) + Fir(features=1)([b]) + c)
demo = Modely("demo", inputs=[a, b, c], outputs=[out]).build()

n = 10
signals = {"a": np.arange(n), "b": 100 + np.arange(n), "c": 1000 + np.arange(n)}
loader = DataLoader(demo, source=signals)
loader

============================ nnodely Model Dataset =============================
Dataset Name:                 demo
Source:                       in-memory dict
Total number of samples:      6
Shape of a:                   (6, 3, 1)
Shape of b:                   (6, 3, 1)
Shape of c:                   (6, 1)

`len(loader)` is the number of samples, and `loader[i]` is sample `i`, a dict
with one array per input:

In [3]:
print("samples:", len(loader))
for i in range(len(loader)):
    sample = loader[i]
    print(i, {name: value.ravel().tolist() for name, value in sample.items()})

samples: 6
0 {'a': [0.0, 1.0, 2.0], 'b': [102.0, 103.0, 104.0], 'c': [1002.0]}
1 {'a': [1.0, 2.0, 3.0], 'b': [103.0, 104.0, 105.0], 'c': [1003.0]}
2 {'a': [2.0, 3.0, 4.0], 'b': [104.0, 105.0, 106.0], 'c': [1004.0]}
3 {'a': [3.0, 4.0, 5.0], 'b': [105.0, 106.0, 107.0], 'c': [1005.0]}
4 {'a': [4.0, 5.0, 6.0], 'b': [106.0, 107.0, 108.0], 'c': [1006.0]}
5 {'a': [5.0, 6.0, 7.0], 'b': [107.0, 108.0, 109.0], 'c': [1007.0]}


Every sample is cut around one **current instant** `t`, the same for all
inputs:

- `a` is `[t-2, t-1, t]`, its 3 samples up to `t`;
- `b` is `[t, t+1, t+2]`, the current sample and the 2 future ones;
- `c` is the value at `t`, without a window.

The first sample is at `t = 2`, the first instant with 3 past values of `a`, and
the last at `t = 7`, the last one with 2 future values of `b`: 10 recorded
samples give `10 - 2 - 2 = 6` samples. The longest past and the longest future
of all the inputs decide how many samples a recording yields.

Each array is laid out as the input is declared - `(T, *D)`, and `(*D)` without
a window. `loader[i]` has **no batch axis**; `as_dict()` stacks every sample
along a batch axis, as the model reads them:

In [4]:
batched = loader.as_dict()
for name, value in batched.items():
    print(f"{name}: {value.shape}")

a: (6, 3, 1)
b: (6, 3, 1)
c: (6, 1)


## Sources

### A dict of arrays

The simplest source: one array per input name, time along the first axis. An
input of several features is `(time, *dim)`, or flat `(time, prod(dim))` in
row-major order. Keys that no input reads are ignored.

The model below, used in the rest of the tutorial, predicts the next position
`y` of a system from its last 3 positions and the last 3 samples of a 2-D force
`u`. The target is the sample **after** the window of `y`, so `y` is declared
with one future sample: `tw=(3, 1)`.

In [5]:
y = Input("y", tw=(3, 1))  # the 3 last positions, and the next one
u = Input("u", dim=2, tw=3)  # the 3 last samples of a 2-D force

The prediction reads only the past of `y`: `TimeRange(0, 3)` takes the first 3
samples of its window, and `TimeSelect(-1)` the last one, the target (see
[Partitioning](07_partitioning.ipynb)). A `Linear` mixes the 2 components of the
force into one, and a `Fir` filters each window into one sample.

In [6]:
force_effect = Fir(features=1)([Linear(features=1, bias=False)([u])])
prediction = Output("y_next", Fir(features=1)([TimeRange(0, 3)(y)]) + force_effect)
model = Modely("plant", inputs=[y, u], outputs=[prediction])
model.minimize("next_position", prediction, TimeSelect(-1)(y))
model.build()

rng = np.random.default_rng(0)
T = 400
time = np.arange(T) * 0.05
force = np.stack([np.sin(time), np.cos(0.5 * time)], axis=1)  # (T, 2)
position = np.convolve(force[:, 0] + 0.5 * force[:, 1], np.ones(5) / 5, mode="same")
position += 0.01 * rng.standard_normal(T)

data = DataLoader(model, source={"y": position, "u": force})
print(len(data), {name: value.shape for name, value in data.as_dict().items()})

397 {'y': (397, 4, 1), 'u': (397, 3, 2)}


Note the force: `(T, 2)`, time first, then the 2 features of `u`.

### A DataFrame, a CSV file, a folder of CSV files

Tables have columns, not input names. `format` maps each input to its columns:
a column name, a column index, or a list of columns for an input of several
features. An input left out of `format` is looked up by its own name, and the
columns no input reads are ignored:

In [7]:
frame = pd.DataFrame(
    {
        "time": time,
        "pos": position,
        "fx": force[:, 0],
        "fy": force[:, 1],
    }
)
columns = {"y": "pos", "u": ["fx", "fy"]}

from_frame = DataLoader(model, source=frame, format=columns)

frame.to_csv(workdir / "run_0.csv", index=False)
from_file = DataLoader(model, source=workdir / "run_0.csv", format=columns)

same = all(
    np.allclose(from_frame.as_dict()[k], from_file.as_dict()[k]) for k in ("y", "u")
)
print(len(from_frame), len(from_file), "same samples:", same)

397 397 same samples: True


CSV files are read by `pandas.read_csv`: `delimiter` and `header` are passed to
it, and a file without a header row is mapped by column index:

In [8]:
frame[["pos", "fx", "fy"]].to_csv(
    workdir / "raw.txt", sep=";", header=False, index=False
)
from_raw = DataLoader(
    model,
    source=workdir / "raw.txt",
    format={"y": 0, "u": [1, 2]},
    delimiter=";",
    header=None,
)
print(len(from_raw))

397


## Several recordings: simulations

Experiments are usually recorded in several runs. A **list** of sources - dicts
or DataFrames - or a **folder** of CSV files is a set of *simulations*. Each is
cut into windows on its own: a window never spans the end of one run and the
start of the next, and the runs may have different lengths.

In [9]:
for index, (start, stop) in enumerate([(0, 150), (150, 280), (280, 400)]):
    frame.iloc[start:stop].to_csv(workdir / f"run_{index + 1}.csv", index=False)

folder = DataLoader(model, source=workdir, format=columns, csv_glob="run_[1-3].csv")
pieces = [
    {"y": position[i:j], "u": force[i:j]} for i, j in [(0, 150), (150, 280), (280, 400)]
]
listed = DataLoader(model, source=pieces)

print("one recording:", len(data), "samples")
print(
    "three runs:   ",
    len(folder),
    "samples (from the folder),",
    len(listed),
    "(from a list)",
)

one recording: 397 samples
three runs:    391 samples (from the folder), 391 (from a list)


Each run loses the first 2 instants (no past for the window of 3) and the last
one (no future for the target): three runs give 6 samples fewer than one long
recording.

## Training and validation data

A `DataLoader` holds one dataset: build one per split. Split the **recordings**,
not the samples, so that no window of the validation set overlaps the training
ones:

In [10]:
split = int(0.8 * T)
train_data = DataLoader(model, source={"y": position[:split], "u": force[:split]})
val_data = DataLoader(model, source={"y": position[split:], "u": force[split:]})
print(len(train_data), len(val_data))

# or with whole runs: two files to train on, one to validate
train_runs = DataLoader(model, source=workdir, format=columns, csv_glob="run_[12].csv")
val_runs = DataLoader(model, source=workdir / "run_3.csv", format=columns)
print(len(train_runs), len(val_runs))

317 77
274 117


`step=k` keeps one sample every `k`: consecutive samples overlap almost
entirely, and sub-sampling a slow signal trains faster on the same information.

In [11]:
print(len(DataLoader(model, source={"y": position, "u": force}, step=4)))

100


## Sequences

An input with a sequence, `Input("s", seq=L, tw=n)`, reads at each sample `L`
consecutive windows: the trajectories of recurrent models are read like this
(see [Recurrent models](10_recurrent_loop.ipynb)). The **last** window of the
sequence ends at the current instant of the other inputs:

In [12]:
s = Input("s", seq=3, tw=2)
k = Input("k", tw=1)
seq_demo = Modely(
    "seq_demo", inputs=[s, k], outputs=[Output("o", Fir(features=1)([s]) + k)]
).build()
seq_loader = DataLoader(seq_demo, source={"s": np.arange(8), "k": 100 + np.arange(8)})
for i in range(len(seq_loader)):
    sample = seq_loader[i]
    print(i, "s:", sample["s"][..., 0].tolist(), " k:", sample["k"].ravel().tolist())

0 s: [[0.0, 1.0], [1.0, 2.0], [2.0, 3.0]]  k: [103.0]
1 s: [[1.0, 2.0], [2.0, 3.0], [3.0, 4.0]]  k: [104.0]
2 s: [[2.0, 3.0], [3.0, 4.0], [4.0, 5.0]]  k: [105.0]
3 s: [[3.0, 4.0], [4.0, 5.0], [5.0, 6.0]]  k: [106.0]
4 s: [[4.0, 5.0], [5.0, 6.0], [6.0, 7.0]]  k: [107.0]


At sample 0 the current instant is `t = 3`: `k` reads `103`, and the last
window of `s` is `[2, 3]`, the two before it `[0, 1]` and `[1, 2]`.

### Dynamic lengths and padding

A sequence of `seq=-1` takes its length from the loader: `seq_length=H` cuts
every recording into samples of `H` steps, and `seq_length="full"` makes each
whole recording one sample. Recordings of different lengths are then
**padded** to the longest, repeating their last step, and `loader.mask` marks
the steps that are real: training and validation leave the padded steps out.

In [13]:
trajectory = Input("trajectory", seq=-1, tw=1)
traj_model = Modely(
    "traj",
    inputs=[trajectory],
    outputs=[Output("scaled", trajectory * Parameter("gain", value=2.0))],
).build()

short_run = {"trajectory": np.arange(4.0)}
long_run = {"trajectory": 10 + np.arange(6.0)}

windows = DataLoader(traj_model, source=[short_run, long_run], seq_length=3)
print(
    "seq_length=3:",
    len(windows),
    "samples of",
    windows.as_dict()["trajectory"].shape[1:],
)

whole = DataLoader(traj_model, source=[short_run, long_run], seq_length="full")
print("full:", whole.as_dict()["trajectory"][..., 0, 0])
print("mask:", whole.mask)

seq_length=3: 6 samples of (3, 1, 1)
full: [[ 0.  1.  2.  3.  3.  3.]
 [10. 11. 12. 13. 14. 15.]]
mask: [[ True  True  True  True False False]
 [ True  True  True  True  True  True]]


## Messy data

A recording too short for the windows raises an error naming it;
`on_short="skip"` drops it instead. Missing cells (`NaN`, empty, `n/a`) are kept
as NaN, with a warning: a NaN reaching the loss makes it NaN, so clean or
interpolate them first.

In [14]:
tiny = {"y": position[:3], "u": force[:3]}
try:
    DataLoader(model, source=[tiny])
except ValueError as error:
    print("ValueError:", error)

kept = DataLoader(model, source=[tiny, {"y": position, "u": force}], on_short="skip")
print("with on_short='skip':", len(kept), "samples")

ValueError: 'simulation 0' has only 3 samples, but the model requires at least 4. Pass on_short='skip' to leave short simulations out.
with on_short='skip': 397 samples


In [15]:
import warnings

holed = position.copy()
holed[[10, 11]] = np.nan
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    DataLoader(model, source={"y": holed, "u": force})
print(caught[0].message)

'simulation 0': input 'y' has 2 n/a values, kept as NaN.


## Data that is already windowed

Data prepared elsewhere - by another tool, or by `as_dict()` - can be passed
with `windowed=True`: each array is then taken as it is, laid out as the
model's tensors `(N, *S, T, *D)`, and nothing is cut or aligned. The round trip
gives back the same arrays:

In [16]:
again = DataLoader(model, source=data.as_dict(), windowed=True)
print(
    len(again),
    all(np.array_equal(again.as_dict()[k], data.as_dict()[k]) for k in ("y", "u")),
)

397 True


## Reading the samples back

| Access | Gives |
|---|---|
| `len(loader)` | the number of samples |
| `loader[i]` | sample `i`: `{name: (*S, T, *D)}`, **no** batch axis |
| `loader.as_dict()` | every sample: `{name: (N, *S, T, *D)}` |
| `loader.get_samples(n, start, simulation)` | `n` consecutive samples of one recording, batched |
| `loader.inputs` | the input names it holds |
| `loader.mask` | which steps are real, with `seq_length="full"` |

A model is run on batched arrays: add the batch axis to a single sample with
`[None]`, or pass `as_dict()` (or `get_samples`) for many.

In [17]:
first = {name: value[None] for name, value in data[0].items()}
print(keras.ops.convert_to_numpy(model(first)["y_next"]).shape)

block = data.get_samples(5, start=10)
print({name: value.shape for name, value in block.items()})
print(keras.ops.convert_to_numpy(model(block)["y_next"]).shape)

(1, 1, 1)
{'y': (5, 4, 1), 'u': (5, 3, 2)}
(5, 1, 1)


The loaders are what `train` and `validate` take:

```python
model.train(train_data, val_data=val_data, epochs=50, batch_size=32)
model.validate(val_data)
```

## Summary

| Task | Code |
|---|---|
| Arrays | `DataLoader(model, source={"x": array, ...})`, time first |
| A table | `source=frame` or `"file.csv"`, `format={"x": "col", "v": ["c1", "c2"]}` |
| A folder of runs | `source="folder", csv_glob="run_*.csv"` |
| Several runs | `source=[run_1, run_2, ...]`: windows never cross them |
| Splits | one loader per split, split by recordings |
| Fewer samples | `step=k` |
| Dynamic sequences | `seq_length=H` or `"full"` (padded, `loader.mask`) |
| Short runs, NaN | `on_short="skip"`; NaN are kept, with a warning |
| Prepared tensors | `windowed=True` |

Next: [Model training](03_model_training.ipynb).